# Especialização de um SLM em português (LoRA)

**Tarefa:** análise de sentimento (3 classes) · **Modelo base:** `neuralmind/bert-base-portuguese-cased` (BERTimbau)

Comparamos o desempenho em duas situações:

- **Baseline** — o modelo base como ponto de partida.
- **LoRA** — o modelo especializado, treinando menos de 1% dos parâmetros.

`modelo → dados → baseline → LoRA → comparação`

### Instalação (só no Google Colab)
No Colab, instala as versões usadas. No seu computador (conda) as libs já existem.

In [43]:
import sys

# Detecta o ambiente: a biblioteca "google.colab" só existe quando rodamos no Colab.
if "google.colab" in sys.modules:
    # Instala as versões EXATAS do projeto. A flag -q (quiet) reduz a saída da instalação.
    !pip install -q "torch==2.6.0" "transformers==4.49.0" "peft==0.14.0" \
        "datasets==3.3.1" "accelerate==1.4.0" "scikit-learn==1.6.1"
    print("Dependências instaladas no Colab.")
else:
    # No seu computador (conda) as libs já estão instaladas -> não faz nada.
    print("Ambiente local: usando as bibliotecas já instaladas.")

Ambiente local: usando as bibliotecas já instaladas.


### Imports, seed e dispositivo
Fixamos a *seed* (mesmo resultado a cada execução) e escolhemos GPU se houver, senão CPU.

In [44]:
import random
import numpy as np
import torch

# --- 1) Fixa a semente dos geradores aleatórios para o resultado ser REPRODUTÍVEL ---
SEED = 42                 # qualquer inteiro fixo serve; 42 é só convenção
random.seed(SEED)         # controla sorteios da biblioteca padrão do Python
np.random.seed(SEED)      # controla sorteios do numpy (ex.: ao embaralhar os dados)
torch.manual_seed(SEED)   # controla sorteios do torch

# --- 2) Escolhe onde o modelo vai rodar ---
HAS_CUDA = torch.cuda.is_available()      # True se houver uma GPU NVIDIA disponível
DEVICE = "cuda" if HAS_CUDA else "cpu"    # usa a GPU se existir; senão, a CPU

print("torch:", torch.__version__, "| GPU?", HAS_CUDA, "| dispositivo:", DEVICE)

torch: 2.6.0 | GPU? False | dispositivo: cpu


### Carregar o modelo e contar os parâmetros
Carregamos o BERTimbau e contamos seus parâmetros, para situá-lo como SLM (pequeno) perante um LLM.

- `MODEL_NAME` — qual modelo baixar do Hugging Face.
- `NUM_LABELS = 3` — número de classes de saída (positivo / neutro / negativo).

In [45]:
import transformers
transformers.logging.set_verbosity_error()   # oculta avisos técnicos do carregamento do modelo

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from IPython.display import Markdown, display

def show(texto):
    # Exibe um texto em markdown montado com os números REAIS da execução (f-string),
    # em vez de valores digitados à mão.
    display(Markdown(texto))

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"   # identificador do BERTimbau no Hugging Face
NUM_LABELS = 3                                         # 3 classes: positivo / neutro / negativo

# 1) Tokenizer: ferramenta que quebra o texto em "tokens" e os converte em números (IDs)
#    que o modelo entende.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2) Modelo: baixa o corpo do BERTimbau e acopla uma cabeça de classificação com NUM_LABELS saídas.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

# 3) Conta os parâmetros (pesos) do modelo.
#    model.parameters() = todos os tensores de pesos; p.numel() = quantos números há em cada um.
total_params = sum(p.numel() for p in model.parameters())
llm_7b = 7_000_000_000   # referência de um LLM "pequeno": 7 bilhões de parâmetros

show(f"""
**Tamanho:** {total_params:,} parâmetros (~{total_params/1e6:.0f} milhões).
Um LLM de 7B é ~{llm_7b/total_params:.0f}× maior → o BERTimbau é um **SLM** (roda em hardware comum).
""")


**Tamanho:** 108,925,443 parâmetros (~109 milhões).
Um LLM de 7B é ~64× maior → o BERTimbau é um **SLM** (roda em hardware comum).


### Carregar os dados
Baixamos um dataset público de sentimento em português e lemos os nomes das classes direto dele.

In [46]:
from datasets import load_dataset

# Baixa o dataset do Hugging Face.
#   1º argumento = nome do dataset; 2º argumento = recorte de idioma ("portuguese").
raw = load_dataset("tyqiangz/multilingual-sentiments", "portuguese")

# features["label"].names = os nomes das classes NA ORDEM oficial (índice 0, 1, 2...).
# Lemos daqui para não "chutar" qual número corresponde a qual sentimento.
class_names = raw["train"].features["label"].names
print("Classes:", class_names)

# Mostra quantos exemplos há em cada parte do dataset (train / validation / test).
print("Tamanho:", {parte: len(raw[parte]) for parte in raw})

# Imprime 3 exemplos (rótulo + começo do texto) só para dar contexto.
for i in range(3):
    ex = raw["train"][i]                                        # i-ésimo exemplo do treino
    print(f'\n[{class_names[ex["label"]]}] {ex["text"][:110]}')  # [classe] + 110 primeiros caracteres

Classes: ['positive', 'neutral', 'negative']
Tamanho: {'train': 1839, 'validation': 324, 'test': 870}

[negative] Que gafe do VídeoShow, colocar uma cena de morte, no dia da morte do Nelson Xavier.  #VideoShowAoVivo

[neutral] Tapoha q prova difícil #MasterChefBR

[positive] A voz da Vanessa da mata é uma coisa maravilhosa #altasHoras


### Amostra e tokenização
Separamos treino/teste e convertemos o texto em números (tokens), que é o que o modelo entende.

- `N_TRAIN` / `N_TEST` — quantas frases usar (treino = estuda; teste = avaliação em frases novas).
- `max_length = 128` — máximo de tokens por frase; o excedente é cortado (frases de sentimento são curtas).

In [47]:
# --- 1) Define o tamanho das amostras, sem passar do que existe no dataset ---
N_TRAIN = min(2000, len(raw["train"]))   # frases de TREINO (o modelo estuda com elas)
N_TEST  = min(500, len(raw["test"]))     # frases de TESTE (avaliação em frases nunca vistas)

# --- 2) Embaralha de forma reprodutível (por causa do seed) e pega as N primeiras ---
#     shuffle(seed=SEED) = embaralhamento determinístico; select(range(N)) = recorta N exemplos.
train_data = raw["train"].shuffle(seed=SEED).select(range(N_TRAIN))
test_data  = raw["test"].shuffle(seed=SEED).select(range(N_TEST))

# --- 3) Função que tokeniza um LOTE de frases ---
def tokenize(batch):
    # truncation=True + max_length=128 -> corta frases com mais de 128 tokens.
    return tokenizer(batch["text"], truncation=True, max_length=128)

# --- 4) Aplica a tokenização a todos os exemplos ---
#     map(..., batched=True) -> processa em lotes (mais rápido).
#     rename_column("label","labels") -> o Trainer do Hugging Face espera a coluna chamada "labels".
train_tok = train_data.map(tokenize, batched=True).rename_column("label", "labels")
test_tok  = test_data.map(tokenize, batched=True).rename_column("label", "labels")

print(f"{len(train_tok)} de treino | {len(test_tok)} de teste")

1839 de treino | 500 de teste


### Baseline: ponto de partida
Medimos o desempenho do modelo base na tarefa, **antes da especialização**, como referência para
medir o ganho do LoRA. Métrica principal: **F1 macro** (justa quando as classes têm tamanhos diferentes).

In [48]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Collator: ao montar cada lote, preenche (padding) as frases até ficarem do mesmo tamanho.
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Função que transforma as previsões do modelo em métricas.
def compute_metrics(pred):
    logits, labels = pred                 # logits = pontuação de cada classe; labels = respostas certas
    preds = np.argmax(logits, axis=-1)    # escolhe a classe de MAIOR pontuação (axis=-1 = última dimensão)
    return {
        "accuracy": accuracy_score(labels, preds),               # fração de acertos
        "f1_macro": f1_score(labels, preds, average="macro"),    # F1 médio por classe (peso igual p/ cada uma)
    }

def avaliar(modelo):
    """Roda o 'modelo' no conjunto de TESTE e devolve accuracy e F1 macro."""
    args = TrainingArguments(
        output_dir="._tmp",               # pasta temporária exigida pela API (não é relevante aqui)
        per_device_eval_batch_size=32,    # quantas frases avaliar por vez
        report_to=[],                     # [] = não envia logs para serviços externos (ex.: Weights & Biases)
        use_cpu=(not HAS_CUDA),           # sem GPU, força CPU (evita erro do backend "MPS" no Mac)
    )
    # Trainer = utilitário do Hugging Face que roda o modelo nos dados.
    #   processing_class=tokenizer informa ao Trainer qual tokenizer usar.
    trainer = Trainer(model=modelo, args=args, data_collator=collator,
                      compute_metrics=compute_metrics, processing_class=tokenizer)
    saida = trainer.predict(test_tok)     # gera as previsões no conjunto de teste
    # saida.predictions = logits; saida.label_ids = rótulos verdadeiros.
    return compute_metrics((saida.predictions, saida.label_ids))

baseline = avaliar(model)   # mede o ponto de partida, antes da especialização
show(f"""
**Baseline:** accuracy {baseline['accuracy']*100:.1f}% · F1 macro {baseline['f1_macro']*100:.1f}%.
É o ponto de partida — vamos melhorar com o LoRA.
""")


**Baseline:** accuracy 33.8% · F1 macro 16.9%.
É o ponto de partida — vamos melhorar com o LoRA.


### LoRA: especialização eficiente
Em vez de retreinar os ~110M de pesos, o LoRA **congela** o modelo e aprende uma pequena correção
`B·A` somada aos pesos: `W' = W + (α/r)·B·A`. Treina menos de 1% dos parâmetros.

- `r = 8` — tamanho das matrizes A e B (capacidade do ajuste; menor = mais econômico).
- `lora_alpha = 16` — escala da correção (na prática vale α/r = 2).
- `lora_dropout = 0.1` — desliga 10% das conexões no treino para não decorar (overfitting).
- `target_modules = ["query","value"]` — onde aplicar o LoRA (peças da atenção do BERT).
- `modules_to_save = ["classifier"]` — a cabeça de classificação é treinada por inteiro.
- `task_type = SEQ_CLS` — a tarefa é classificar a frase inteira.

In [49]:
from peft import LoraConfig, get_peft_model, TaskType

# Recria o modelo base para o treino começar do ponto de partida.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

# Configuração do LoRA — cada parâmetro explicado:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,          # tipo da tarefa: classificação da sequência (a frase inteira)
    r=8,                                 # "posto": tamanho das matrizes A e B (menor = menos parâmetros)
    lora_alpha=16,                       # escala da correção; ela é multiplicada por alpha/r (= 16/8 = 2)
    lora_dropout=0.1,                    # desliga 10% das conexões no treino p/ evitar decorar (overfitting)
    target_modules=["query", "value"],   # camadas onde o LoRA entra: "query" e "value" da atenção
    modules_to_save=["classifier"],       # além do LoRA, treina por completo a cabeça de classificação
)

# get_peft_model: "embrulha" o modelo com os adaptadores LoRA e CONGELA o resto automaticamente.
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()   # imprime quantos parâmetros ficaram treináveis

# Conta manualmente (só para a explicação):
#   requires_grad=True indica os pesos que SERÃO ajustados no treino.
treinaveis = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora_model.parameters())
show(f"""
**Treináveis:** {treinaveis:,} de {total:,} (**{100*treinaveis/total:.2f}%**). O resto fica congelado.
""")

trainable params: 297,219 || all params: 109,222,662 || trainable%: 0.2721



**Treináveis:** 297,219 de 109,222,662 (**0.27%**). O resto fica congelado.


### Treino do LoRA
Ajusta só as matrizes A/B e a cabeça.

- `num_train_epochs = 5` — quantas vezes passa por todos os dados (poucas treinam de menos).
- `per_device_train_batch_size = 16` — frases lidas antes de cada ajuste dos pesos.
- `learning_rate = 2e-4` — tamanho do passo de ajuste (o LoRA tolera valores maiores).
- `eval_strategy = "no"` — não avalia no meio do treino (acelera em CPU).

⏱️ CPU: ~20-30 min · GPU (Colab T4): segundos.

In [50]:
# Configurações do treino — cada parâmetro explicado:
args = TrainingArguments(
    output_dir="._lora",                # pasta onde o Trainer salva arquivos/checkpoints temporários
    num_train_epochs=5,                 # nº de passagens por TODO o treino
    per_device_train_batch_size=16,     # nº de frases lidas antes de cada ajuste dos pesos
    per_device_eval_batch_size=32,      # nº de frases por vez na avaliação
    learning_rate=2e-4,                 # "tamanho do passo" de cada ajuste (o LoRA tolera valores maiores)
    eval_strategy="no",                 # "no" = não avalia durante o treino (avaliamos só no fim) -> + rápido em CPU
    logging_steps=50,                   # imprime o progresso (o erro/"loss") a cada 50 passos
    report_to=[],                       # não envia logs para serviços externos
    seed=SEED,                          # semente p/ reprodutibilidade do treino
    use_cpu=(not HAS_CUDA),             # sem GPU, treina na CPU
)

# Monta o Trainer juntando: modelo LoRA, configurações, dados, collator e métricas.
trainer = Trainer(
    model=lora_model, args=args,
    train_dataset=train_tok,            # dados de TREINO (tokenizados)
    eval_dataset=test_tok,              # dados usados na avaliação
    data_collator=collator,             # monta os lotes com padding
    compute_metrics=compute_metrics,    # função que calcula accuracy e F1
    processing_class=tokenizer,
)

trainer.train()        # executa o treino: ajusta as matrizes A, B e a cabeça de classificação
print("Treino concluído.")

{'loss': 1.0982, 'grad_norm': 1.7093496322631836, 'learning_rate': 0.00018260869565217392, 'epoch': 0.43478260869565216}
{'loss': 1.0843, 'grad_norm': 1.6732325553894043, 'learning_rate': 0.00016521739130434784, 'epoch': 0.8695652173913043}
{'loss': 1.0589, 'grad_norm': 2.440600633621216, 'learning_rate': 0.00014782608695652173, 'epoch': 1.3043478260869565}
{'loss': 1.0016, 'grad_norm': 2.915776491165161, 'learning_rate': 0.00013043478260869567, 'epoch': 1.7391304347826086}
{'loss': 0.9139, 'grad_norm': 2.652259111404419, 'learning_rate': 0.00011304347826086956, 'epoch': 2.1739130434782608}
{'loss': 0.8463, 'grad_norm': 3.0966241359710693, 'learning_rate': 9.565217391304348e-05, 'epoch': 2.608695652173913}
{'loss': 0.8289, 'grad_norm': 4.093550682067871, 'learning_rate': 7.82608695652174e-05, 'epoch': 3.0434782608695654}
{'loss': 0.834, 'grad_norm': 3.9551033973693848, 'learning_rate': 6.086956521739131e-05, 'epoch': 3.4782608695652173}
{'loss': 0.7628, 'grad_norm': 3.497185707092285, 

### Comparação final
Avaliamos o LoRA no mesmo teste e comparamos com o baseline. O texto é gerado com os números reais.

In [51]:
lora_result = avaliar(lora_model)   # avalia o modelo JÁ especializado (LoRA) no conjunto de teste

# Diferença de F1 macro entre LoRA e baseline, em pontos percentuais (x100):
ganho_lora = (lora_result["f1_macro"] - baseline["f1_macro"]) * 100
ganho_acc  = (lora_result["accuracy"] - baseline["accuracy"]) * 100

show(f"""
| Fase | O que treina | Accuracy | F1 macro |
|------|--------------|---------:|---------:|
| Baseline | — | {baseline['accuracy']*100:.1f}% | {baseline['f1_macro']*100:.1f}% |
| **LoRA** | adaptadores + cabeça (<1%) | **{lora_result['accuracy']*100:.1f}%** | **{lora_result['f1_macro']*100:.1f}%** |

**Evolução do baseline para o LoRA:** F1 macro **{ganho_lora:+.1f} pontos** e accuracy **{ganho_acc:+.1f} pontos**,
treinando **menos de 1%** dos parâmetros.
""")


| Fase | O que treina | Accuracy | F1 macro |
|------|--------------|---------:|---------:|
| Baseline | — | 33.8% | 16.9% |
| **LoRA** | adaptadores + cabeça (<1%) | **70.0%** | **70.1%** |

**Evolução do baseline para o LoRA:** F1 macro **+53.2 pontos** e accuracy **+36.2 pontos**,
treinando **menos de 1%** dos parâmetros.
